# QC for final assemblies

1.) Input QC
- PoreC
- ReadLengths UL
- ReadLengths HQ_herro

2.) Assembly QC

In [45]:
library(tidyverse)
library(ggplot2)
library(gt)

In [46]:
samples <- c(
    "GE-MED-T2T00",
    "GE-MED-T2T04"
)

## 1.) Input QC

In [ ]:
source("../scripts/02_plot_read_stats.R")

types <- c("HQ_herro.50x", "UL.70x")

dt_input_qc <- expand.grid(sample = samples, type = types) %>%
    mutate(path = paste0("../../assembly/input_qc/", sample, "/", sample, ".", type, "/read_stats.txt"))

dt_input_qc

In [ ]:
dt_l <- list()
for (i in 1:nrow(dt_input_qc)) {
    print(paste("Processing", dt_input_qc$sample[i]))
    dt_l[[dt_input_qc$path[i]]] <- process_sequencing_file(dt_input_qc$path[i])
} 

In [49]:
for (i in 1:length(dt_l)) {
    dt_l[[i]]$summary_stats$path <- names(dt_l)[i]
    dt_l[[i]]$read_length_density$path <- names(dt_l)[i]
    dt_l[[i]]$quality_density$path <- names(dt_l)[i]
    dt_l[[i]]$density_2d$path <- names(dt_l)[i]
}
dt_stats <- bind_rows(lapply(dt_l, function(x) x$summary_stats)) %>%
    inner_join(dt_input_qc, by = c("path" = "path")) %>%
    select(c(-sample_name))
dt_read_length_density <- bind_rows(lapply(dt_l, function(x) x$read_length_density)) %>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_quality_density <- bind_rows(lapply(dt_l, function(x) x$quality_density))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))
dt_density_2d <- bind_rows(lapply(dt_l, function(x) x$density_2d))%>%
        inner_join(dt_input_qc, by = c("path" = "path"))%>%
    select(c(-sample_name))


In [ ]:
dt_stats

In [ ]:
dt_stats_reduced <- dt_stats %>%
    select(sample, type, total_bases, N50, yield_above_100kb, median_quality, reads_above_1MB)
dt_stats_reduced

### PoreC QC, using results from wf-porec pipeline

In [52]:
#todo

## Assembly QC Results

In [ ]:
source("../scripts/13_process_assembly_qc.R")
dt <- read_tsv("/mnt/storage3b/projects/no_ngsd/ahthapp1_T2T_ONT/assembly/qc/qc_samples.tsv") %>%
    process_qc_table %>%
    mutate(haplotype = ifelse(is.na(haplotype), "both", haplotype))
head(dt)

In [ ]:
dt_qc_table <- dt %>%
  filter() %>%
  mutate(haplotype = ifelse(haplotype == "assembly", "both", haplotype)) %>%
  group_by(asm_name, metric, haplotype) %>%
  summarise(value = sum(value, na.rm = TRUE), .groups = 'drop') %>%
  mutate(
    formatted_value = case_when(

      metric %in% c("Error Rate") ~ paste0(round(value * 100, 3), "%"),
      TRUE ~ as.character(round(value, 2))
    )
  ) %>%
  select(-value) %>%  # Remove original value column
  pivot_wider(
    names_from = metric, 
    values_from = formatted_value
  )

write_tsv(dt_qc_table, "../../doc/tables/qc_samples_table.tsv")

dt_qc_table  %>%
  knitr::kable()

In [70]:
# Select and prepare data
qc_selected <- dt_qc_table %>%
  select(
    asm_name,
    haplotype,
    `Assembly Length`,
    n_contigs_over_10mb,
    n_contigs,
    `% of Assembly covered by Ref`,
    `% of Ref covered by Assembly`,
    `Genome Completeness`, # This seems to be a direct percentage value (e.g. 94.62 means 94.62%)
    `Missing Multi-Copy Genes (%)`,
    `Quality Value`, # This is likely a Phred-like score, not percentage
    `Overall Switch Rate (%)`, # This is a proportion (0.05 = 5%)
    `Number of T2T chromosomes`,
    total_n_count,
    total_gaps
  ) %>%
  mutate(
    # Clean column names for easier use in gt (optional but good practice)
    # Here we will use backticks later, so not strictly necessary for this script
    
    # Ensure other numeric columns are numeric (read_tsv is good, but explicit is safer)
    # Note: `Overall Switch Rate (%)` is already a proportion like 0.05
    # `Genome Completeness` values like 94.62 are treated as direct percentage values
    # `Quality Value` is a score
    across(c(`Assembly Length`, `Quality Value`, `Overall Switch Rate (%)`,`% of Assembly covered by Ref`, `% of Ref covered by Assembly`, `Genome Completeness`, total_n_count), as.numeric),
    across(c( `Number of T2T chromosomes`), as.integer),
    
    # Factor haplotype for desired order in table rows
    haplotype = factor(haplotype, levels = c("haplotype1", "haplotype2", "both"))
  ) %>%
  arrange(asm_name, haplotype)



In [ ]:
# Create the gt table
qc_gt_table <- qc_selected %>% 
  gt(rowname_col = "haplotype", groupname_col = "asm_name") %>%
  tab_header(
    title = md("**Assembly QC Metrics**"),
    subtitle = "Selected key metrics"
  ) %>%
  cols_label(
    `Assembly Length` = html("Assembly<br>Length"),
    `Genome Completeness` = html("Genome<br>Completeness"),
    `Overall Switch Rate (%)` = html("Switch<br>Error Rate"),
    `Number of T2T chromosomes` = html("T2T<br>Chroms"),
    `total_n_count` = html("Total Ns"),
    `total_gaps` = html("Total Gaps"),
    `n_contigs`="n Contigs",
    `n_contigs_over_10mb`="n Contigs >10Mbp",
  ) %>%
  # Formatting numeric columns
  fmt_number(
    columns = `Assembly Length`,
    decimals = 2,
    scale_by = 1/1e9, # Convert to Gbp
    pattern = "{x} Gbp"
  ) %>%
   fmt_number(
    columns = `Number of T2T chromosomes`,
    decimals = 0
  ) %>%
  fmt_number(
    columns = `% of Assembly covered by Ref`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
  fmt_number( 
    columns = `% of Ref covered by Assembly`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
   fmt_number( # For Genome Completeness (e.g., BUSCO score, already in percent value)
    columns = `Genome Completeness`,
    decimals = 2,
    pattern = "{x}%" # Append % sign
  ) %>%
  # Handle missing values
  fmt_missing(
    columns = everything(),
    missing_text = "—" # Display NAs as a dash
  ) %>%
  # Align columns
  cols_align(
    align = "center",
    columns = where(is.numeric) # Center numeric columns
  ) %>%
  cols_align(
    align = "left", # The rowname_col (haplotype) defaults to left, which is good
    columns = `asm_name` # The groupname_col (asm_name) also default to left.
  ) %>%
  # Add some styling
  tab_options(
    table.border.top.color = "black",
    table.border.bottom.color = "black",
    table.width = pct(90), # Make table width 90% of container
    heading.title.font.size = px(20),
    heading.subtitle.font.size = px(15),
    column_labels.border.bottom.color = "black",
    column_labels.font.weight = "bold",
    row_group.font.weight = "bold",
    row_group.background.color = "#f0f0f0", # Light grey for group headers
    table_body.hlines.color = "#D3D3D3" # Light grey horizontal lines in body
  ) %>%
  tab_source_note(
    source_note = "Data from qc_samples_table.txt. Gbp: Giga base pairs, Mbp: Mega base pairs."
  )

gts <- function(gt_table){
   gt:::as.tags.gt_tbl(gt_table)
}

# Print the table
qc_gt_table %>% gts

### Table for Hifiasm

In [ ]:
source("../scripts/13_process_assembly_qc.R")
dt_hifiasm <- read_tsv("/mnt/storage3b/projects/no_ngsd/ahthapp1_T2T_ONT/assembly/qc/qc_hifiasm.tsv") %>%
    process_qc_table %>%
    mutate(haplotype = ifelse(is.na(haplotype), "both", haplotype))
head(dt_hifiasm)

In [ ]:
dt_qc_table_hifiasm <- dt_hifiasm %>%
  filter() %>%
  mutate(haplotype = ifelse(haplotype == "assembly", "both", haplotype)) %>%
  group_by(asm_name, metric, haplotype) %>%
  summarise(value = sum(value, na.rm = TRUE), .groups = 'drop') %>%
  mutate(
    formatted_value = case_when(

      metric %in% c("Error Rate") ~ paste0(round(value * 100, 3), "%"),
      TRUE ~ as.character(round(value, 2))
    )
  ) %>%
  select(-value) %>%  # Remove original value column
  pivot_wider(
    names_from = metric, 
    values_from = formatted_value
  )

write_tsv(dt_qc_table_hifiasm, "../../doc/tables/qc_samples_table.tsv")

dt_qc_table_hifiasm  %>%
  knitr::kable()

In [79]:
# Select and prepare data
qc_selected_hifiasm <- dt_qc_table_hifiasm %>%
  select(
    asm_name,
    haplotype,
    `Assembly Length`,
    n_contigs_over_10mb,
    n_contigs,
    `% of Assembly covered by Ref`,
    `% of Ref covered by Assembly`,
    `Genome Completeness`, # This seems to be a direct percentage value (e.g. 94.62 means 94.62%)
    `Missing Multi-Copy Genes (%)`,
    `Quality Value`, # This is likely a Phred-like score, not percentage
    `Overall Switch Rate (%)`, # This is a proportion (0.05 = 5%)
    `Number of T2T chromosomes`,
    total_n_count,
    total_gaps
  ) %>%
  mutate(
    # Clean column names for easier use in gt (optional but good practice)
    # Here we will use backticks later, so not strictly necessary for this script
    
    # Ensure other numeric columns are numeric (read_tsv is good, but explicit is safer)
    # Note: `Overall Switch Rate (%)` is already a proportion like 0.05
    # `Genome Completeness` values like 94.62 are treated as direct percentage values
    # `Quality Value` is a score
    across(c(`Assembly Length`, `Quality Value`, `Overall Switch Rate (%)`,`% of Assembly covered by Ref`, `% of Ref covered by Assembly`, `Genome Completeness`, total_n_count), as.numeric),
    across(c( `Number of T2T chromosomes`), as.integer),
    
    # Factor haplotype for desired order in table rows
    haplotype = factor(haplotype, levels = c("haplotype1", "haplotype2", "both"))
  ) %>%
  arrange(asm_name, haplotype)

In [ ]:
# Create the gt table
qc_gt_table_hifiasm <- qc_selected_hifiasm %>% 
  gt(rowname_col = "haplotype", groupname_col = "asm_name") %>%
  tab_header(
    title = md("**Assembly QC Metrics**"),
    subtitle = "Selected key metrics"
  ) %>%
  cols_label(
    `Assembly Length` = html("Assembly<br>Length"),
    `Genome Completeness` = html("Genome<br>Completeness"),
    `Overall Switch Rate (%)` = html("Switch<br>Error Rate"),
    `Number of T2T chromosomes` = html("T2T<br>Chroms"),
    `total_n_count` = html("Total Ns"),
    `total_gaps` = html("Total Gaps"),
    `n_contigs`="n Contigs",
    `n_contigs_over_10mb`="n Contigs >10Mbp",
  ) %>%
  # Formatting numeric columns
  fmt_number(
    columns = `Assembly Length`,
    decimals = 2,
    scale_by = 1/1e9, # Convert to Gbp
    pattern = "{x} Gbp"
  ) %>%
   fmt_number(
    columns = `Number of T2T chromosomes`,
    decimals = 0
  ) %>%
  fmt_number(
    columns = `% of Assembly covered by Ref`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
  fmt_number( 
    columns = `% of Ref covered by Assembly`,
    decimals = 2,
    scale_by = 100,
    pattern = "{x}%" # Append % sign
  ) %>%
   fmt_number( # For Genome Completeness (e.g., BUSCO score, already in percent value)
    columns = `Genome Completeness`,
    decimals = 2,
    pattern = "{x}%" # Append % sign
  ) %>%
  # Handle missing values
  fmt_missing(
    columns = everything(),
    missing_text = "—" # Display NAs as a dash
  ) %>%
  # Align columns
  cols_align(
    align = "center",
    columns = where(is.numeric) # Center numeric columns
  ) %>%
  cols_align(
    align = "left", # The rowname_col (haplotype) defaults to left, which is good
    columns = `asm_name` # The groupname_col (asm_name) also default to left.
  ) %>%
  # Add some styling
  tab_options(
    table.border.top.color = "black",
    table.border.bottom.color = "black",
    table.width = pct(90), # Make table width 90% of container
    heading.title.font.size = px(20),
    heading.subtitle.font.size = px(15),
    column_labels.border.bottom.color = "black",
    column_labels.font.weight = "bold",
    row_group.font.weight = "bold",
    row_group.background.color = "#f0f0f0", # Light grey for group headers
    table_body.hlines.color = "#D3D3D3" # Light grey horizontal lines in body
  ) %>%
  tab_source_note(
    source_note = "Data from qc_samples_table.txt. Gbp: Giga base pairs, Mbp: Mega base pairs."
  )

gts <- function(gt_table){
   gt:::as.tags.gt_tbl(gt_table)
}

# Print the table
qc_gt_table_hifiasm %>% gts